In [4]:
import os
import requests
from typing import TypedDict

from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, START, END

In [5]:
with open(r"C:\Users\user\Desktop\Shubham-AI\Shared\ey-ai-upskill-10-main\ey-ai-upskill-10-main\key-vault\huggingface\openai\api.key") as f:
    openai_api_key = f.read().strip()
os.environ["OPENAI_API_KEY"] = openai_api_key

In [6]:
CIS_URL = r"http://127.0.0.1:8000/ask"

In [7]:
with open(r"C:\Users\user\Desktop\Shubham-AI\Shared\ey-ai-upskill-10-main\ey-ai-upskill-10-main\key-vault\huggingface\nvd\api.key") as f:
    nvd_api_key = f.read().strip()
NVD_API_KEY = nvd_api_key

In [8]:
MODEL = "gpt-4.1-mini"

In [9]:
llm = ChatOpenAI(model=MODEL, temperature=0)

In [10]:
import requests

In [13]:
@tool
def ask_cis_policy(query: str,
                   endpoint: str = "http://127.0.0.1:8000/ask") -> str:
    """
    Calls the CIS Policy Intelligence API and returns only the answer.
    Args:
        query: User query.
        endpoint: FastAPI endpoint URL.
    Returns:
        Answer string.
    Raises:
        requests.HTTPError: If the API returns an error.
        ValueError: If the response does not contain an 'answer' field.
    """
    response = requests.post(
        endpoint,
        json={"query": query},
        timeout=60
    )
    response.raise_for_status()
    data = response.json()
    if "answer" not in data:
        raise ValueError("Response does not contain 'answer'.")
    return data["answer"]

In [14]:
answer = ask_cis_policy.invoke("What are the password policies?")
print(answer)

1. **Answer**  
The password policies recommended in the CIS document include several critical settings aimed at enhancing security. These policies focus on password history, complexity, length, and expiration. The main recommendations are:  
   - **Enforce Password History**: Set to 24 or more passwords to prevent reuse.  
   - **Maximum Password Age**: Set to 365 or fewer days, but not 0, to ensure regular password changes.  
   - **Minimum Password Age**: Set to 1 or more days to prevent immediate password changes.  
   - **Minimum Password Length**: Set to 14 or more characters to enhance password strength.  
   - **Password Complexity Requirements**: Enabled to ensure passwords include a mix of character types.  
   - **Relax Minimum Password Length Limits**: Enabled for specific scenarios.  
   - **Store Passwords Using Reversible Encryption**: Disabled to enhance security.

2. **Relevant CIS Controls**  
- **5.2 Use Unique Passwords**: Use unique passwords for all enterprise ass

In [15]:
@tool
def lookup_cve(keyword:str)->str:
    """Query the NVD API for CVEs matching a keyword."""
    print("[TOOL]CVE Tool Activated")
    url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
    headers = {"apiKey": NVD_API_KEY} if NVD_API_KEY else {}
    params = {"keywordSearch": keyword, "resultsPerPage": 3}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=20)
        r.raise_for_status()
        data = r.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return "No CVEs found."
        lines = []
        for v in vulns:
            c = v["cve"]
            lines.append(f"{c['id']}: {c['descriptions'][0]['value'][:180]}")
        return "\n".join(lines)
    except Exception as e:
        return f"CVE lookup failed: {e}"

In [16]:
lookup_cve.invoke("SMB")

[TOOL]CVE Tool Activated


'CVE-1999-1387: Windows NT 4.0 SP2 allows remote attackers to cause a denial of service (crash), possibly via malformed inputs or packets, such as those generated by a Linux smbmount command that \nCVE-1999-0225: Windows NT 4.0 allows remote attackers to cause a denial of service via a malformed SMB logon request in which the actual data size does not match the specified size.\nCVE-1999-0495: A remote attacker can gain access to a file system using ..  (dot dot) when accessing SMB shares.'

In [ ]:
IPCONFIG TOOL

In [17]:
import subprocess

@tool
def ipconfig_tool() -> str:
    """
    Returns the Windows network configuration using the ipconfig command.
    """
    try:
        result = subprocess.run(
            ["ipconfig"],
            capture_output=True,
            text=True,
            check=True,
            shell=True
        )

        return result.stdout

    except subprocess.CalledProcessError as e:
        return f"Error executing ipconfig:\n{e.stderr}"

In [18]:
print(ipconfig_tool.invoke({}))


Windows IP Configuration


Ethernet adapter Ethernet0:

   Connection-specific DNS Suffix  . : deepcloud.in
   Link-local IPv6 Address . . . . . : fe80::4ab1:3490:f80:b836%5
   IPv4 Address. . . . . . . . . . . : 10.33.5.56
   Subnet Mask . . . . . . . . . . . : 255.255.255.0
   Default Gateway . . . . . . . . . : 10.33.5.1



In [ ]:
###3. Agents Layer

In [19]:
planner = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a planner. Decide weather RAG and CVE lookup is required"
)

In [20]:
retrieval_agent = create_agent(
    model=llm,
    tools=[ask_cis_policy],
    system_prompt="Use the CIS policy tool to retrieve CIS benchmark guidance. Use the provided tools mandatorily"
)

In [21]:
threat_agent = create_agent(
    model=llm,
    tools=[lookup_cve],
    system_prompt="Use CVE lookup tool when threat intelligence is needed. Use the provided tools mandatorily"
)

In [22]:
validator_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="Validate the evidence and produce a concise final answer"
)

In [ ]:
Note: you can test the agent independently using <agent>.invoke({})

In [ ]:
###4. state

In [23]:
class CyberState(TypedDict):
    query: str
    plan: str
    rag: str
    cves: str
    final: str

In [ ]:
### 5. Node

In [24]:
def planner_node(state: CyberState):
    r = planner.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"plan": str(r)}

In [25]:
def rag_node(state: CyberState):
    r = retrieval_agent.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"rag": str(r)}

In [26]:
def cve_node(state: CyberState):
    r = threat_agent.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"cves": str(r)}

In [27]:
def validator_node(state: CyberState):

    prompt = f"""
User Query:
{state["query"]}

Plan:
{state.get("plan", "")}

RAG:
{state.get("rag", "")}

CVEs:
{state.get("cves", "")}

SCORE:
validation score

Produce the final validated response.
Also, give a score from 0 to 10
"""
    r = validator_agent.invoke({
        "messages":[
            {"role":"user", "content":prompt}
        ]
    })
    return {"final": r}

In [ ]:
6. WorkFlow

In [28]:
graph = StateGraph(CyberState)

graph.add_node("planner", planner_node)
graph.add_node("rag", rag_node)
graph.add_node("cve", cve_node)
graph.add_node("validator", validator_node)

graph.add_edge(START, "planner")
graph.add_edge("planner", "rag")
graph.add_edge("rag", "cve")
graph.add_edge("cve", "validator")
graph.add_edge("validator", END)

In [29]:
app = graph.compile()

In [ ]:
7. Execute

In [30]:
result = app.invoke({
    "query": "How can I harden Windows SMB services against ransomware?"
})

In [31]:
result

{'query': 'How can I harden Windows SMB services against ransomware?',
 'plan': "{'messages': [HumanMessage(content='How can I harden Windows SMB services against ransomware?', additional_kwargs={}, response_metadata={}, id='8fa7b75a-5d48-4109-93d1-c51331c214c9'), AIMessage(content='RAG (Retrieval-Augmented Generation) and CVE (Common Vulnerabilities and Exposures) lookup would be useful here to provide the most up-to-date and specific security measures and known vulnerabilities related to Windows SMB services.\\n\\nI will proceed with RAG and CVE lookup to gather detailed and current information on hardening Windows SMB services against ransomware.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 37, 'total_tokens': 108, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_toke

In [32]:
print(result["final"]["messages"][-1].content)

Final Answer:

To harden Windows SMB services against ransomware, implement the following key security measures:

1. **Disable SMBv1**: This outdated protocol is vulnerable and commonly exploited by ransomware (e.g., EternalBlue). Disable SMBv1 via Group Policy:
   ```
   Computer Configuration\Policies\Administrative Templates\MS Security Guide\Configure SMB v1 server = Disabled
   ```

2. **Require SMB Encryption**: Enforce encryption for SMB traffic to protect data in transit:
   ```
   Computer Configuration\Policies\Administrative Templates\Network\Lanman Workstation\Require Encryption = Enabled
   ```

3. **Enable Auditing for Unencrypted SMB Traffic**: Monitor and log SMB connections that do not use encryption to detect potential threats:
   ```
   Computer Configuration\Policies\Administrative Templates\Network\Lanman Server\Audit client does not support encryption = Enabled
   ```

4. **Disable Unnecessary SMB-Related Services**: Reduce the attack surface by turning off SMB se

In [33]:
query = "Recommend CIS settings for Windows RDP and include recent vulnerabilities that administrators should patch."
result = app.invoke({
    "query": query
})

[TOOL]CVE Tool Activated
[TOOL]CVE Tool Activated


In [34]:
print(result["final"]["messages"][-1].content)

Final Answer:

**Recommended CIS Settings for Windows Remote Desktop Protocol (RDP):**

1. **Enable Network Level Authentication (NLA):**  
   Require users to authenticate before a remote session is established.  
   *Group Policy path:*  
   `Computer Configuration\Policies\Administrative Templates\Windows Components\Remote Desktop Services\Remote Desktop Session Host\Security\Require user authentication for remote connections by using Network Level Authentication` (set to Enabled).

2. **Limit RDP Access to Authorized Users:**  
   Restrict which user accounts or groups can log on via Remote Desktop Services.

3. **Use Strong Encryption:**  
   Configure RDP to use the highest encryption level available (e.g., TLS 1.2 or higher).

4. **Configure Session Timeouts and Reconnection Policies:**  
   Set limits on idle session time and control reconnection behavior.

5. **Enable Account Lockout Policies:**  
   Prevent brute force attacks by locking accounts after a number of failed logi